# Recombination with GRG

**Note:** To add sample nodes (e.g., offspring from recombination), use grgl from the `grg_modify_improve` branch:

```bash
git clone -b grg_modify_improve --recursive https://github.com/aprilweilab/grgl.git
cd grgl && python setup.py bdist_wheel && pip install --force-reinstall dist/*.whl
```

The `MutableGRG.set_samples(sample_nodes)` API lets you add new sample nodes by passing the full list of sample node IDs (including the new offspring).

In [1]:
import numpy as np

NEGATIVE_NODE_IDS = []  
def get_breakpoints(N, p=0.1, min_breakpoints=3):
    """
    Return breakpoint positions. Ensures at least min_breakpoints in interior
    so recombination is less likely to be entire interval from one parent.
    """
    bp = np.where(np.random.binomial(1, p, N))[0]
    # Ensure at least one interior breakpoint to avoid entire-interval inheritance
    interior = np.arange(1, N)
    while len(bp) < min_breakpoints and len(interior) >= min_breakpoints:
        bp = np.unique(np.concatenate([bp, np.random.choice(interior, min(min_breakpoints, len(interior)), replace=False)]))
        bp = np.sort(bp)
    print(f"Generated breakpoints: {bp}")
    return bp

def recombination_intervals(h1, h2, N):
    """
    Returns list of segments: [(source_parent_id, end_coord), ...]
    """
    bp = get_breakpoints(N)
    start = np.random.binomial(1, 0.5, 1)[0]
    parents = [h1, h2]
    
    segments = []
    for i, K in enumerate(bp):
        segments.append((parents[(start + i) % 2], K))
    segments.append((parents[(start + len(bp)) % 2], N))
    return segments

In [2]:
from pygrgl import MutableGRG, Mutation, save_grg, grg_to_cyto_json
import pygrgl

def create_simple_grg():
    grg = MutableGRG(4, 1, True) 
    
    # Create 4 internal nodes (4, 5, 6, 7)
    grg.make_node()  # node 4 (has m6, m7)
    grg.make_node()  # node 5 (has m8)
    grg.make_node()  # node 6 (has m9)
    grg.make_node()  # node 7 (has m10, m11)
    
    # Add edges (parent -> child) 
    grg.connect(6, 0)  # 6 -> 0
    grg.connect(6, 5)  # 6 -> 5
    grg.connect(7, 5)  # 7 -> 5
    grg.connect(7, 4)  # 7 -> 4
    grg.connect(5, 1)  # 5 -> 1
    grg.connect(5, 4)  # 5 -> 4
    grg.connect(4, 1)  # 4 -> 1
    grg.connect(4, 2)  # 4 -> 2
    grg.connect(4, 3)  # 4 -> 3
    
    # Add mutations (m1-m11)
    grg.add_mutation(Mutation(0, "A", "G"), 1)   # m1 on node 1
    grg.add_mutation(Mutation(1, "C", "T"), 0)   # m2 on node 0
    grg.add_mutation(Mutation(2, "G", "A"), 0)   # m3 on node 0
    grg.add_mutation(Mutation(3, "T", "C"), 2)   # m4 on node 2
    grg.add_mutation(Mutation(4, "A", "T"), 3)   # m5 on node 3
    grg.add_mutation(Mutation(5, "C", "G"), 4)   # m6 on node 4
    grg.add_mutation(Mutation(6, "G", "C"), 4)   # m7 on node 4
    grg.add_mutation(Mutation(7, "T", "A"), 5)   # m8 on node 5
    grg.add_mutation(Mutation(8, "A", "C"), 6)   # m9 on node 6
    grg.add_mutation(Mutation(9, "C", "A"), 7)  # m10 on node 7
    grg.add_mutation(Mutation(10, "G", "T"), 7)  # m11 on node 7
    
    return grg

# Create and save the GRG
simple_grg = create_simple_grg()


print("=== Simple GRG Created ===")
print(f"Nodes: {simple_grg.num_nodes}")
print(f"Edges: {simple_grg.num_edges}")
print(f"Mutations: {simple_grg.num_mutations}")
print(f"Samples: {simple_grg.get_sample_nodes()}")
print()

# Display structure
print("Node details:")
for node_id in range(simple_grg.num_nodes):
    parents = simple_grg.get_up_edges(node_id)
    children = simple_grg.get_down_edges(node_id)
    muts = simple_grg.get_mutations_for_node(node_id)
    mut_names = [f"m{m+1}" for m in muts]  # Just show mutation names
    is_sample = "[SAMPLE]" if simple_grg.is_sample(node_id) else ""
    print(f"  Node {node_id}: parents={parents}, children={children}, muts={mut_names} {is_sample}")

# Save to file
save_grg(simple_grg, "simple_example.grg")
print()
print("Saved to simple_example.grg")

cyto_data = grg_to_cyto_json(simple_grg)
print()
print("Cytoscape JSON representation:")
print(f"  Nodes: {len(cyto_data['nodes'])}")
print(f"  Edges: {len(cyto_data['edges'])}")
for node in cyto_data['nodes']:
    print(f"    {node['data']}")
for edge in cyto_data['edges']:
    print(f"    {edge['data']}")

=== Simple GRG Created ===
Nodes: 8
Edges: 9
Mutations: 11
Samples: [0, 1, 2, 3]

Node details:
  Node 0: parents=[6], children=[], muts=['m2', 'm3'] [SAMPLE]
  Node 1: parents=[5, 4], children=[], muts=['m1'] [SAMPLE]
  Node 2: parents=[4], children=[], muts=['m4'] [SAMPLE]
  Node 3: parents=[4], children=[], muts=['m5'] [SAMPLE]
  Node 4: parents=[7, 5], children=[1, 2, 3], muts=['m6', 'm7'] 
  Node 5: parents=[6, 7], children=[1, 4], muts=['m8'] 
  Node 6: parents=[], children=[0, 5], muts=['m9'] 
  Node 7: parents=[], children=[5, 4], muts=['m10', 'm11'] 

Saved to simple_example.grg

Cytoscape JSON representation:
  Nodes: 8
  Edges: 9
    {'id': 'n0', 'label': 'id=0, S, mutations({1, 2})', 'is_sample': 'True'}
    {'id': 'n1', 'label': 'id=1, S, mutations({0})', 'is_sample': 'True'}
    {'id': 'n2', 'label': 'id=2, S, mutations({3})', 'is_sample': 'True'}
    {'id': 'n3', 'label': 'id=3, S, mutations({4})', 'is_sample': 'True'}
    {'id': 'n4', 'label': 'id=4, mutations({5, 6})',

In [3]:
GRG_FILE = "simple_example.grg" 

In [4]:
class NonDuplicationRecombination:
    """
    Non-duplication GRG recombination algorithm.
    
    Key operations:
    - Path Compression: Skip nodes with no relevant mutations
    - Pruning: Stop when ancestry is disjoint from query interval
    - Bubble Insertion: Split nodes when partial mutation inheritance is needed
    """

    debug_mode = False  # Set to True to enable debug prints
    
    # def __init__(self, grg):
    #     self.grg = grg
    #     self.genome_length = grg.bp_range[1]  # l_max
    #     self.original_bp_range = grg.bp_range  # Store original bp_range for reference
    #     self._span_cache = {}               # NEW: Caches the full ancestor span
    #     self._ancestral_coverage_cache = {} # NEW: Caches the ancestral coverage
    #     self._mutation_cache = {}
    #     self.NEGATIVE_NODE_IDS = []
    #     self._modified_nodes = set()        # Track which nodes were modified for selective cache invalidation
    #     self._pending_bubbles = []          # Queue of bubble operations to defer until after traversal
    
    def __init__(self, grg):
        self.grg = grg
        self.genome_length = grg.bp_range[1]
        self.original_bp_range = grg.bp_range
        self._mutation_cache = {}
        self.NEGATIVE_NODE_IDS = []
        self._modified_nodes = set()
        self._pending_bubbles = []
        self._pending_sample_removals = set() # NEW: Defer C++ boundary crossing
        
        self._prefetch_all_mutations()
        
        self.span_cache = [False] * self.grg.num_nodes
        self.anc_cov_cache = [False] * self.grg.num_nodes

    def _prefetch_all_mutations(self):
        self._mutation_cache = {i: [] for i in range(self.grg.num_nodes)}
        self._pos_cache = {} # NEW: Raw float lists for bisect

        for node_id, mut_id in self.grg.get_node_mutation_pairs():
            mut = self.grg.get_mutation_by_id(mut_id)
            self._mutation_cache[node_id].append((mut_id, mut.position))
            
        for node_id in self._mutation_cache:
            if self._mutation_cache[node_id]:
                self._mutation_cache[node_id].sort(key=lambda x: x[1])
                # Lock in a pure list of floats for O(log M) searching
                self._pos_cache[node_id] = [m[1] for m in self._mutation_cache[node_id]]

    # def _get_node_mutations(self, node_id):
    #     """Get mutation positions for a node (cached, sorted by position)."""
    #     if node_id not in self._mutation_cache:
    #         mut_ids = self.grg.get_mutations_for_node(node_id)
    #         mutations = []
    #         for mut_id in mut_ids:
    #             mut = self.grg.get_mutation_by_id(mut_id)
    #             mutations.append((mut_id, mut.position))
    #         # Sort by position for fast interval queries using binary search
    #         mutations.sort(key=lambda x: x[1])
    #         self._mutation_cache[node_id] = mutations
    #     return self._mutation_cache[node_id]

    def _get_node_mutations(self, node_id):
        """Get mutation positions for a node (cached, sorted by position)."""
        # Because we eager loaded, this is almost always an instant native Python lookup
        if node_id not in self._mutation_cache:
            # Fallback ONLY used for newly created bubble nodes mid-traversal
            mut_ids = self.grg.get_mutations_for_node(node_id)
            mutations = []
            for mut_id in mut_ids:
                mut = self.grg.get_mutation_by_id(mut_id)
                mutations.append((mut_id, mut.position))
            # Sort by position for fast interval queries using binary search
            mutations.sort(key=lambda x: x[1])
            self._mutation_cache[node_id] = mutations
            
        return self._mutation_cache[node_id]
    
    def _get_mutations_in_interval(self, node_id, L, R):
        """Get mutations on node_id that fall within [L, R) using binary search (O(log N) time)."""
        node_muts = self._get_node_mutations(node_id)
        if not node_muts:
            return []
        
        # Extract positions for binary search
        positions = [pos for _, pos in node_muts]
        
        # Find the range [L, R) using binary search
        left_idx = bisect.bisect_left(positions, L)
        right_idx = bisect.bisect_left(positions, R)
        
        return node_muts[left_idx:right_idx]
    
    def _has_connected_descendant(self, node_id, connected, visited=None):
        """True if any descendant of node_id (via down edges) is in connected."""
        if visited is None:
            visited = set()
        if node_id in visited:
            return False
        visited.add(node_id)
        for child in self.grg.get_down_edges(node_id):
            if child in connected:
                return True
            if self._has_connected_descendant(child, connected, visited):
                return True
        return False

    def _has_connected_ancestor(self, node_id, connected, visited=None):
        """True if any ancestor of node_id (via up edges) is in connected."""
        if visited is None:
            visited = set()
        if node_id in visited:
            return False
        visited.add(node_id)
        for parent in self.grg.get_up_edges(node_id):
            if parent in connected:
                return True
            if self._has_connected_ancestor(parent, connected, visited):
                return True
        return False

    # def _get_ancestral_coverage(self, node_id):
    #     """Compute the ancestral interval coverage Iu (mutations in ancestors ONLY)."""
    #     # 1. Check cache
    #     if node_id in self._ancestral_coverage_cache:
    #         return self._ancestral_coverage_cache[node_id]

    #     parents = self.grg.get_up_edges(node_id)
    #     if not parents:
    #         self._ancestral_coverage_cache[node_id] = None
    #         return None
            
    #     min_pos, max_pos = float('inf'), float('-inf')
    #     visited = set() 
        
    #     for parent in parents:
    #         parent_span = self._get_node_and_ancestor_span(parent, visited)
    #         if parent_span:
    #             min_pos = min(min_pos, parent_span[0])
    #             max_pos = max(max_pos, parent_span[1])
        
    #     # 2. Format result and cache it
    #     result = None if min_pos == float('inf') else (min_pos, max_pos + 1)
        
    #     self._ancestral_coverage_cache[node_id] = result
    #     return result

    # def _get_node_and_ancestor_span(self, node_id, visited):
    #     """Recursive helper: gets span of mutations for this node and its ancestors."""
    #     # 1. Check cache
    #     if node_id in self._span_cache:
    #         return self._span_cache[node_id]
            
    #     if node_id in visited:
    #         return None
    #     visited.add(node_id)
        
    #     min_pos, max_pos = float('inf'), float('-inf')
        
    #     # 2. Check this node's mutations
    #     node_muts = self._get_node_mutations(node_id)
    #     if node_muts:
    #         min_pos = min(min_pos, min(pos for _, pos in node_muts))
    #         max_pos = max(max_pos, max(pos for _, pos in node_muts))
        
    #     # 3. Recurse to parents
    #     for parent in self.grg.get_up_edges(node_id):
    #         anc_span = self._get_node_and_ancestor_span(parent, visited)
    #         if anc_span:
    #             min_pos = min(min_pos, anc_span[0])
    #             max_pos = max(max_pos, anc_span[1])
                
    #     # 4. Format result and cache it
    #     result = None if min_pos == float('inf') else (min_pos, max_pos)
        
    #     self._span_cache[node_id] = result
    #     return result

    def _get_mutation_range(self, node_id, L, R):
        """Returns (start_idx, end_idx) for mutations in [L, R) in O(log M)."""
        # Fallback for bubbles created mid-run
        if node_id not in self._pos_cache:
            node_muts = self._get_node_mutations(node_id)
            self._pos_cache[node_id] = [m[1] for m in node_muts]
            
        positions = self._pos_cache[node_id]
        if not positions:
            return 0, 0
            
        return bisect.bisect_left(positions, L), bisect.bisect_left(positions, R)

    def _get_node_and_ancestor_span(self, node_id):
        """Lazy recursive helper using array memoization."""
        # 1. Instant Array Cache Hit
        if self.span_cache[node_id] is not False:
            return self.span_cache[node_id]
            
        min_pos, max_pos = float('inf'), float('-inf')
        
        # 2. Check this node's mutations
        node_muts = self._get_node_mutations(node_id)
        if node_muts:
            min_pos = node_muts[0][1]
            max_pos = node_muts[-1][1]
        
        # 3. Recurse to parents (only runs once per node)
        for parent in self.grg.get_up_edges(node_id):
            anc_span = self._get_node_and_ancestor_span(parent)
            if anc_span:
                min_pos = min(min_pos, anc_span[0])
                max_pos = max(max_pos, anc_span[1])
                
        # 4. Format result and lock it in the array
        result = None if min_pos == float('inf') else (min_pos, max_pos)
        self.span_cache[node_id] = result
        return result

    def _get_ancestral_coverage(self, node_id):
        """Compute Iu lazily using array memoization."""
        if self.anc_cov_cache[node_id] is not False:
            return self.anc_cov_cache[node_id]

        parents = self.grg.get_up_edges(node_id)
        if not parents:
            self.anc_cov_cache[node_id] = None
            return None
            
        min_pos, max_pos = float('inf'), float('-inf')
        
        for parent in parents:
            p_span = self._get_node_and_ancestor_span(parent)
            if p_span:
                min_pos = min(min_pos, p_span[0])
                max_pos = max(max_pos, p_span[1])
                
        result = None if min_pos == float('inf') else (min_pos, max_pos + 1)
        self.anc_cov_cache[node_id] = result
        return result   
    
    def _extract_bubble(self, node_id, relevant_mut_ids, offspring_id, interval):
        """
        Queue a bubble node creation to be deferred until after traversal.
        
        Instead of immediately modifying the graph, this method queues the bubble
        operation to be applied after traversal completes. This prevents mid-traversal
        cache invalidations and graph disruptions.
        
        Creates a new node v that:
        - Contains the relevant mutations (moved from node_id)
        - Becomes a parent of node_id
        - Becomes a parent of the offspring
        
        Args:
            node_id: The node to split
            relevant_mut_ids: Mutation IDs to move to the bubble
            offspring_id: The offspring node to connect
            interval: The interval [L, R) being inherited
            
        Returns:
            The bubble node ID (created immediately for graph structure)
        """

        # Create new bubble node immediately (needed for edge connections)
        bubble_id = self.grg.make_node()

        # Expand Lazy arrays for the new node safely
        while len(self.span_cache) <= bubble_id:
            self.span_cache.append(False)
            self.anc_cov_cache.append(False)         
        
        # Add connections immediately (needed for graph structure)
        self.grg.connect(bubble_id, node_id)
        self.grg.connect(bubble_id, -offspring_id)
        
        # Queue mutations to be moved later (deferred)
        # This avoids cascading cache invalidations during traversal
        self._pending_bubbles.append({
            'node_id': node_id,
            'bubble_id': bubble_id,
            'relevant_mut_ids': relevant_mut_ids
        })
        
        # Track which nodes were modified for selective cache invalidation (to be done after traversal)
        self._modified_nodes.add(node_id)        # Will lose mutations
        self._modified_nodes.add(bubble_id)      # New bubble node
        # Also track parents since node_id's composition will change
        for parent in self.grg.get_up_edges(node_id):
            self._modified_nodes.add(parent)
        
        # Don't invalidate caches yet - let traversal continue on stable graph
        # They'll be cleared after all bubbles are applied
        
        return bubble_id
    
    # def _recurse_attach(self, node_id, offspring_id, L, R, visited=None, connected=None):
    #     """
    #     Recursively attach ancestry to offspring for interval [L, R).
        
    #     Algorithm 2: RecurseAttach
        
    #     Handles:
    #     - Full Coverage (I ⊆ Iu): Ancestors fully cover query
    #     - Disjoint (I ∩ Iu = ∅): Pruning - stop traversal
    #     - Partial Overlap: Decomposition - recurse on intersection
        
    #     Args:
    #         node_id: Current ancestor node
    #         offspring_id: The new offspring node
    #         L, R: Query interval [L, R)
    #         visited: Set of already visited nodes to avoid cycles
    #         connected: Set of nodes already connected to offspring (avoid duplicates)
    #     """
    #     if L >= R:
    #         return
        
    #     if visited is None:
    #         visited = set()
    #     if connected is None:
    #         connected = set()
        
    #     if node_id in visited:
    #         return
    #     visited.add(node_id)
        
    #     # # Get mutations and coverage info
    #     # all_muts = self._get_node_mutations(node_id)
    #     # relevant_muts = self._get_mutations_in_interval(node_id, L, R)
        
    #     # all_mut_ids = set(mut_id for mut_id, _ in all_muts)
    #     # relevant_mut_ids = set(mut_id for mut_id, _ in relevant_muts)

    #     # # Check mutation relevance
    #     # has_all_relevant = (relevant_mut_ids == all_mut_ids) and len(all_mut_ids) > 0
    #     # has_no_relevant = len(relevant_mut_ids) == 0
    #     # has_partial_relevant = len(relevant_mut_ids) > 0 and relevant_mut_ids != all_mut_ids

    #     # Get mutations and coverage info
    #     all_muts = self._get_node_mutations(node_id)
    #     relevant_muts = self._get_mutations_in_interval(node_id, L, R)
        
    #     num_all = len(all_muts)
    #     num_rel = len(relevant_muts)

    #     # NEW: FAST O(1) mathematical checks instead of O(M) set creations
    #     has_all_relevant = (num_all > 0) and (num_rel == num_all)
    #     has_no_relevant = (num_rel == 0)
    #     has_partial_relevant = (num_rel > 0) and (num_rel < num_all)
        
    #     # Get ancestral coverage
    #     Iu = self._get_ancestral_coverage(node_id)
        
    #     if self.debug_mode:
    #         print(f"Visiting node {node_id}: all_muts={all_muts}, relevant_muts={relevant_muts}, Iu={Iu}")
        
    #     # Determine coverage status
    #     if Iu is None:
    #         if self.debug_mode:
    #             print("No ancestral mutations - treating as root node")
    #         if has_all_relevant:
    #             if self.debug_mode:
    #                 print("Root Node Case 1: All relevant mutations - connect directly")
    #             # if (node_id not in connected and
    #             #     not self._has_connected_descendant(node_id, connected)):
    #             if node_id not in connected:
    #                 self.grg.connect(node_id, -offspring_id)
    #                 connected.add(node_id)

    #                 # Queue sample removal instead of crossing into C++ immediately
    #                 self._pending_sample_removals.add(node_id)

    #                 # Remove sample status if connecting an original sample node to avoid confusion with new offspring samples
    #                 # current_samples = list(self.grg.get_sample_nodes())
    #                 # if node_id in current_samples:
    #                 #     current_samples.remove(node_id)
    #                 #     self.grg.set_samples(current_samples)

    #             elif self.debug_mode:
    #                 print(f"Node {node_id} already connected or has connected relatives, skipping direct connection")
            
    #         if has_partial_relevant:
    #             if self.debug_mode:
    #                 print("Root Node Case 2: Partial relevant mutations - create bubble and connect")
                
    #             # Generate the list of IDs ONLY if a bubble is actually needed
    #             rel_mut_ids = [mut_id for mut_id, _ in relevant_muts]
    #             bubble_id = self._extract_bubble(node_id, rel_mut_ids, offspring_id, (L, R))
    #             connected.add(bubble_id)
                
    #             # bubble_id = self._extract_bubble(node_id, list(relevant_mut_ids), 
    #             #                                         offspring_id, (L, R))
    #             # connected.add(bubble_id)
    #             # connected.add(node_id)
            
    #         return # Stop - lineage fully resolved

    def _recurse_attach(self, node_id, offspring_id, L, R, visited=None, connected=None):
        if L >= R: return
        
        if visited is None: visited = set()
        if connected is None: connected = set()
        
        if node_id in visited: return
        visited.add(node_id)
        
        # 1. Get ONLY the range indices (O(log M))
        left, right = self._get_mutation_range(node_id, L, R)
        num_rel = right - left
        num_all = len(self._pos_cache.get(node_id, []))

        has_all_relevant = (num_all > 0) and (num_rel == num_all)
        has_no_relevant = (num_rel == 0)
        has_partial_relevant = (num_rel > 0) and (num_rel < num_all)
        
        Iu = self._get_ancestral_coverage(node_id)
        
        # Scenario: Root / Terminal Node
        if Iu is None:
            if has_all_relevant:
                if node_id not in connected:
                    self.grg.connect(node_id, -offspring_id)
                    connected.add(node_id)
                    self._pending_sample_removals.add(node_id)
            
            if has_partial_relevant:
                # ONLY NOW do we extract the IDs for the bubble
                rel_mut_ids = [m[0] for m in self._mutation_cache[node_id][left:right]]
                bubble_id = self._extract_bubble(node_id, rel_mut_ids, offspring_id, (L, R))
                connected.add(bubble_id)
            return
        
        # Check if disjoint (Pruning - Scenario 2)
        # I ∩ Iu = ∅
        ancestral_disjoint = R <= Iu[0] or L >= Iu[1]

        # Check interval coverage
        full_coverage = (Iu[0] >= L and Iu[1] <= R)  # Iu ⊆ I (node is fully consumed)
        
        if has_all_relevant and full_coverage:
            if self.debug_mode:
                print("Case 1: Full coverage with all relevant mutations - connect directly")
            # if (node_id not in connected and
            #     not self._has_connected_descendant(node_id, connected)):
            if node_id not in connected:
                self.grg.connect(node_id, -offspring_id)
                connected.add(node_id)

                # Queue sample removal instead of crossing into C++ immediately
                self._pending_sample_removals.add(node_id)

                # Remove sample status if connecting an original sample node to avoid confusion with new offspring samples
                # current_samples = list(self.grg.get_sample_nodes())
                # if node_id in current_samples:
                #     current_samples.remove(node_id)
                #     self.grg.set_samples(current_samples)

            elif self.debug_mode:
                print(f"Node {node_id} already connected or has connected relatives, skipping direct connection")
            return  # Stop - lineage fully resolved
        
        if has_no_relevant and ancestral_disjoint:
            if self.debug_mode:
                print("Case 2: No relevant mutations and disjoint ancestry - prune")
            # End search no relevant mutations to be found
            return
        
        if has_no_relevant and not ancestral_disjoint:
            if self.debug_mode:
                print("Case 3: No relevant mutations but not disjoint ancestry - path compression")
            # Path Compression - bypass this node
            # Recurse on parents
            parents = self.grg.get_up_edges(node_id)
            
            # If there's partial overlap with ancestry, we need to adjust the interval for the parents to the intersection of [L, R) and Iu
            newL = max(L, Iu[0]) 
            newR = min(R, Iu[1]) 
            L, R = newL, newR
            if L >= R: #avoids useless parent calls on degenerate overlap at boundaries 
                    return

            for parent in parents:
                self._recurse_attach(parent, offspring_id, L, R, visited, connected)
            return

        if has_partial_relevant or has_all_relevant:
            if self.debug_mode:
                print("Case 4: Partial/full relevant mutations - need to create bubble and maybe recurse upwards")
            # Cases where we have relevant mutations (partial or full) but not full coverage - need to create bubble and maybe recurse upwards
            
            # Generate the list of IDs ONLY if a bubble is actually needed
            rel_mut_ids = [m[0] for m in self._mutation_cache[node_id][left:right]]
            bubble_id = self._extract_bubble(node_id, rel_mut_ids, offspring_id, (L, R))
            connected.add(bubble_id)
            
            # bubble_id = self._extract_bubble(node_id, list(relevant_mut_ids), 
            #                                         offspring_id, (L, R))
            # connected.add(bubble_id)
            # connected.add(node_id)  # Treat split node as covered to avoid connecting ancestors

            if not ancestral_disjoint:
                # If there's partial overlap with ancestry, we need to adjust the interval for the parents to the intersection of [L, R) and Iu
                newL = max(L, Iu[0])
                newR = min(R, Iu[1])
                L, R = newL, newR
                if L >= R: #avoids useless parent calls on degenerate overlap at boundaries 
                    return

                parents = self.grg.get_up_edges(node_id)
                for parent in parents:
                    self._recurse_attach(parent, offspring_id, L, R, visited, connected)
            
            return
    
    # def _apply_pending_bubbles(self):
    #     """Apply all deferred bubble modifications after traversal completes.
        
    #     This batches all mutation movements and ensures the graph structure
    #     remains stable during traversal, avoiding cascading cache invalidations.
    #     """
    #     for bubble_op in self._pending_bubbles:
    #         node_id = bubble_op['node_id']
    #         bubble_id = bubble_op['bubble_id']
    #         relevant_mut_ids = bubble_op['relevant_mut_ids']
            
    #         # Move relevant mutations from node to bubble (Algorithm 3: v.M <- Mrel; u.M <- u.M \ Mrel)
    #         for mut_id in relevant_mut_ids:
    #             mut = self.grg.get_mutation_by_id(mut_id)
                
    #             # Add mutation to bubble node
    #             self.grg.add_mutation(mut, bubble_id)
    #             # Remove mutation from original node
    #             self.grg.remove_mutation(mut_id, node_id)
        
    #     # Clear the queue
    #     self._pending_bubbles.clear()

    def _apply_pending_bubbles(self):
        """Apply all deferred bubble modifications and sample removals after traversal."""
        for bubble_op in self._pending_bubbles:
            node_id = bubble_op['node_id']
            bubble_id = bubble_op['bubble_id']
            relevant_mut_ids = bubble_op['relevant_mut_ids']
            
            for mut_id in relevant_mut_ids:
                mut = self.grg.get_mutation_by_id(mut_id)
                self.grg.add_mutation(mut, bubble_id)
                self.grg.remove_mutation(mut_id, node_id)
        
        self._pending_bubbles.clear()

        # NEW: Process pending sample removals in one single fast batch
        if self._pending_sample_removals:
            current_samples = set(self.grg.get_sample_nodes())
            current_samples.difference_update(self._pending_sample_removals)
            self.grg.set_samples(list(current_samples))
            self._pending_sample_removals.clear()
    
    # def _clear_modified_caches(self):
    #     """Clear only the caches for nodes that were modified during recombination.
        
    #     This is much more efficient than clearing all caches, as most nodes in a large
    #     graph are unmodified and their cached data remains valid for the next offspring.
    #     """
    #     for node_id in self._modified_nodes:
    #         self._mutation_cache.pop(node_id, None)
    #         self._span_cache.pop(node_id, None)
    #         self._ancestral_coverage_cache.pop(node_id, None)
    #     self._modified_nodes.clear()
    #     self._precompute_spans_iteratively()  # Recompute spans for modified nodes and their ancestors

    def _clear_modified_caches(self):
        for node_id in self._modified_nodes:
            self._mutation_cache.pop(node_id, None)
            self._pos_cache.pop(node_id, None) # Clear the position cache too!
            if node_id < len(self.span_cache):
                self.span_cache[node_id] = False
                self.anc_cov_cache[node_id] = False
        self._modified_nodes.clear()
    
    def recombine(self, haplotype_A, haplotype_B, breakpoint):
        """
        Generate offspring through recombination.
        
        Creates a new offspring node inheriting:
        - [0, breakpoint) from haplotype_A
        - [breakpoint, genome_length) from haplotype_B
        
        Args:
            haplotype_A: Node ID of first parent
            haplotype_B: Node ID of second parent  
            breakpoint: Crossover position
            
        Returns:
            Node ID of the new offspring
        """
        # Ensure pending bubbles are cleared from previous operations
        self._pending_bubbles.clear()
        
        # Create offspring node
        offspring_id = self.grg.make_node(negative=True)
        
        # Track connected nodes to avoid duplicate edges
        connected = set()
        
        # Inherit from haplotype_A for [0, breakpoint)
        self._recurse_attach(haplotype_A, offspring_id, 0, breakpoint, 
                            visited=None, connected=connected)
        
        # Inherit from haplotype_B for [breakpoint, genome_length)
        self._recurse_attach(haplotype_B, offspring_id, breakpoint, self.genome_length,
                            visited=None, connected=connected)
        
        # Apply all deferred bubble modifications after traversal completes
        self._apply_pending_bubbles()

        # NEW: Clear caches and rebuild DP arrays for the next generation
        self._clear_modified_caches()
        
        #self.grg.bp_range = self.original_bp_range  # Ensure bp_range is updated to reflect the full genome length
        
        if offspring_id not in self.NEGATIVE_NODE_IDS:
            self.NEGATIVE_NODE_IDS.append(offspring_id)
        return -(self.NEGATIVE_NODE_IDS.index(offspring_id) + 1)
    
    def recombine_multi(self, segments):
        """
        Generate offspring from multiple segments.
        
        Args:
            segments: List of (parent_node_id, interval_end) tuples
                     where intervals are implicit from previous end
                     
        Returns:
            Node ID of the new offspring
        """
        # Ensure pending bubbles are cleared from previous operations
        self._pending_bubbles.clear()
        
        # Create offspring node
        offspring_id = self.grg.make_node(negative=True)
        
        # Track connected nodes to avoid duplicate edges (maintained across all segments)
        connected = set()
        
        # Process each segment
        start = 0
        for parent_id, end in segments:
            if end > start:
                if self.debug_mode:
                    print("BREAK")
                # Reuse connected set across segments to avoid duplicate connections
                self._recurse_attach(parent_id, offspring_id, start, end,
                                    visited=None, connected=connected)
            start = end
        
        # Apply all deferred bubble modifications after traversal completes
        # This prevents mid-traversal cache invalidations and keeps the graph stable
        self._apply_pending_bubbles()

        # NEW: Clear caches and rebuild DP arrays for the next generation
        self._clear_modified_caches()
        
        #self.grg.bp_range = self.original_bp_range  # Ensure bp_range is updated to reflect the full genome length
        
        if offspring_id not in self.NEGATIVE_NODE_IDS:
            self.NEGATIVE_NODE_IDS.append(offspring_id)
        return -(self.NEGATIVE_NODE_IDS.index(offspring_id) + 1)

In [5]:
import pygrgl
from pygrgl import load_mutable_grg, grg_to_cyto_json
import re

try:
    from pygrgl.display import grg_to_cyto, DAG_STYLE
    from ipycytoscape import CytoscapeWidget
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    DAG_STYLE = None
    CytoscapeWidget = None

if 'NEGATIVE_NODE_IDS' not in globals():
    NEGATIVE_NODE_IDS = []

def display_grg(grg, title="GRG", negative_node_ids=None):
    """Display GRG - uses widget if available, otherwise text."""
    neg_ids = negative_node_ids if negative_node_ids is not None else NEGATIVE_NODE_IDS
    cyto = grg_to_cyto_json(grg, start_from=grg.get_root_nodes())

    for n in cyto['nodes']:
        node_id = int(n['data']['id'][1:])
        if node_id in neg_ids:
            disp = -(neg_ids.index(node_id) + 1)
            n['data']['label'] = n['data']['label'].replace(f'id={node_id}', f'id={disp}', 1)
        
        # Record positions of mutations for this node to display in label
        positions = []
        for mut_id in grg.get_mutations_for_node(node_id):
            mut = grg.get_mutation_by_id(mut_id)
            positions.append(mut.position)

        pattern = r"mutations\(\{.*?\}\)"
        positions_str = ", ".join(str(n) for n in positions)
        # Build the replacement string using your new contents
        replacement = f"mutations({{{positions_str}}})"

        # Replace the whole mutations({ ... }) with the new one
        new_str = re.sub(pattern, replacement, n['data']['label'])
        n['data']['label'] = new_str
        
    if WIDGETS_AVAILABLE:
        try:
            widget = CytoscapeWidget()
            widget.graph.add_graph_from_json(cyto, directed=True)
            widget.set_style(DAG_STYLE)
            widget.set_layout(name="dagre")
            display(widget)
            return
        except Exception as e:
            print(f"Widget display failed: {e}")
    
    print(f"\n{'='*50}")
    print(f" {title}")
    print(f"{'='*50}")
    print(f"Nodes: {len(cyto['nodes'])}, Edges: {len(cyto['edges'])}")
    print("\nNodes:")
    for n in cyto['nodes']:
        d = n['data']
        marker = "[S]" if d.get('is_sample') == 'True' else "   "
        print(f"  {marker} {d['id']}: {d['label']}")
    print("\nEdges (parent → child):")
    for e in cyto['edges']:
        src, tgt = e['data']['source'], e['data']['target']
        try:
            tgt_id = int(tgt[1:])
            tgt_display = f'n{-(neg_ids.index(tgt_id) + 1)}' if tgt_id in neg_ids else tgt
        except ValueError:
            tgt_display = tgt
        print(f"      {src} → {tgt_display}")
    print(f"{'='*50}\n")

grg = create_simple_grg()

def print_grg_state(grg, title="GRG State"):
    """Helper to print GRG state."""
    print(f"=== {title} ===")
    print(f"Nodes: {grg.num_nodes}, Edges: {grg.num_edges}, Mutations: {grg.num_mutations}")
    print(f"Samples: {grg.get_sample_nodes()}")
    print(f"Genome range: {grg.bp_range}")
    print()
    all_nodes = pygrgl.get_topo_order(grg, pygrgl.TraversalDirection.DOWN, grg.get_root_nodes())
    for node_id in all_nodes:
        display_id = -(NEGATIVE_NODE_IDS.index(node_id) + 1) if node_id in NEGATIVE_NODE_IDS else node_id
        up = grg.get_up_edges(node_id)
        down = grg.get_down_edges(node_id)
        muts = grg.get_mutations_for_node(node_id)
        mut_names = [f"m{m+1}" for m in muts]  # Just show mutation names
        is_sample = " [SAMPLE]" if grg.is_sample(node_id) else ""
        print(f"  Node {display_id}: parents={up}, children={down}, muts={mut_names}{is_sample}")

# Show initial state
#print_grg_state(grg, "BEFORE Recombination")
display_grg(grg, "BEFORE Recombination")

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

In [6]:
recomb = NonDuplicationRecombination(grg)

haplotype_A = 2  
haplotype_B = 3  
breakpoint = 3 
segments = []
segments.append((haplotype_A, breakpoint))
segments.append((haplotype_B, grg.bp_range[1]))

print("=" * 60)
print("RECOMBINATION")
print("=" * 60)
print(f"Haplotype A: {haplotype_A}")
print(f"Haplotype B: {haplotype_B}")
print(f"Breakpoint: {breakpoint}")
print()
print(f"Offspring inherits [0, {breakpoint}) from haplotype {haplotype_A}")
print(f"Offspring inherits [{breakpoint}, {grg.bp_range[1]}) from haplotype {haplotype_B}")
print()

# Perform recombination
offspring_id = recomb.recombine(haplotype_A, haplotype_B, breakpoint)

try:
    raw_id = NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
    current_samples = list(grg.get_sample_nodes())
    grg.set_samples(current_samples + [raw_id])
except AttributeError:
    pass

print(f"Created offspring node: {offspring_id}")
print()

# Show state after recombination
print_grg_state(grg, "AFTER Recombination")
display_grg(grg, "AFTER Recombination")

RECOMBINATION
Haplotype A: 2
Haplotype B: 3
Breakpoint: 3

Offspring inherits [0, 3) from haplotype 2
Offspring inherits [3, 11) from haplotype 3



NameError: name 'bisect' is not defined

In [ ]:
def verify_offspring_mutations(recomb, offspring_id, haplotype_A, haplotype_B, segments):
    """
    Verify that offspring has correct mutations based on recombination.
    
    Expected mutations:
    - From haplotype_A: all ancestral mutations with position < breakpoint
    - From haplotype_B: all ancestral mutations with position >= breakpoint
    """
    def get_all_ancestral_mutations(node_id, visited=None):
        """Get all mutation IDs from node and its ancestors."""
        if visited is None:
            visited = set()
        if node_id in visited:
            return []
        visited.add(node_id)
        
        mutations = list(recomb.grg.get_mutations_for_node(node_id))
        
        for parent in recomb.grg.get_up_edges(node_id):
            mutations.extend(get_all_ancestral_mutations(parent, visited))
        
        return mutations
    
    def mut_name(mut_id):
        return f"m{mut_id + 1}"
    
    def get_position(mut_id):
        return recomb.grg.get_mutation_by_id(mut_id).position
    
    genome_length = recomb.grg.bp_range[1]
    
    # Get ancestral mutations for both haplotypes
    muts_A = get_all_ancestral_mutations(haplotype_A)
    muts_B = get_all_ancestral_mutations(haplotype_B)

    # Expected mutations for offspring based on segments
    expected_from_A = []
    expected_from_B = []

    # Determine expected mutations based on segments and their source haplotypes
    start = 0
    for parent, end in segments:
        if parent == haplotype_A:
            for m in muts_A:
                if get_position(m) >= start and get_position(m) < end:
                    expected_from_A.append(m)
        else:
            for m in muts_B:
                if get_position(m) >= start and get_position(m) < end:
                    expected_from_B.append(m)
        start = end
    
    expected_muts = set(expected_from_A + expected_from_B)
    expected_from_A.sort()
    expected_from_B.sort()
    
    # Get actual offspring mutations (traversing ancestry)
    grg_node_id = recomb.NEGATIVE_NODE_IDS[abs(offspring_id) - 1] if offspring_id < 0 else offspring_id
    actual_muts = set(get_all_ancestral_mutations(grg_node_id))

    muts_A.sort()
    muts_B.sort()
    
    if recomb.debug_mode:
        print("=== Mutation Verification ===")
        print(f"Haplotype A (node {haplotype_A}) mutations: {[get_position(m) for m in muts_A]}")
        print(f"Haplotype B (node {haplotype_B}) mutations: {[get_position(m) for m in muts_B]}")
        print()
        print(f"Expected from A: {[get_position(m) for m in expected_from_A]}")
        print(f"Expected from B: {[get_position(m) for m in expected_from_B]}")
        print()
        print(f"Expected: {sorted([get_position(m) for m in expected_muts])}")
        print(f"Actual:   {sorted([get_position(m) for m in actual_muts])}")
        print()
    
    if expected_muts == actual_muts:
        print("✓ Offspring mutations are CORRECT!")
        return True
    else:
        missing = expected_muts - actual_muts
        extra = actual_muts - expected_muts
        if missing:
            print(f"✗ Missing: {[get_position(m) for m in missing]}")
        if extra:
            print(f"✗ Extra: {[get_position(m) for m in extra]}")
        return False

# Verify the recombination
print("=== Verification ===")
breakpoints = []
breakpoints.append(breakpoint)
recombs = NonDuplicationRecombination(grg)
verify_offspring_mutations(recombs, offspring_id, haplotype_A, haplotype_B, segments)

In [ ]:
def generate_offspring(grg, h1, h2, num_offspring, N=None):
    """
    Generate a recombined offspring from two parent haplotypes.
    
    Uses the recombination_intervals function to generate random
    crossover breakpoints, then applies non-duplication recombination.
    
    Args:
        grg: MutableGRG instance
        h1: First parent haplotype node ID
        h2: Second parent haplotype node ID
        N: Genome length (defaults to grg.bp_range[1])
        
    Returns:
        Tuple of (offspring_node_id, segments)
    """
    if N is None:
        N = grg.bp_range[1]
        print(f"Using genome length from GRG: {N}")
    
    offspring_ids = []
    for i in range(num_offspring):
        # Get recombination segments
        segments = recombination_intervals(h1, h2, N)
    
        # Create recombination handler
        recomb = NonDuplicationRecombination(grg)
    
        # Perform recombination
        offspring_id = recomb.recombine_multi(segments)
        raw_id = recomb.NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
        offspring_ids.append(raw_id)

        if recomb.debug_mode:
            print()
            print(f"Generated offspring node: {offspring_id}")
            print(f"Samples (including offspring): {grg.get_sample_nodes()}")
            print(f"Segments inherited: {segments}")
            print()

            print("Segment breakdown:")
            start = 0
            for parent, end in segments:
                print(f"  [{start}, {end}): from parent {parent}")
                start = end
            
            print("")
        verify_offspring_mutations(recomb, offspring_id, h1, h2, segments)

    # try:
    #     current_samples = list(grg.get_sample_nodes())
    #     raw_id = NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
    #     grg.set_samples(current_samples + [raw_id])
    # except AttributeError:
    #     pass  
    
    return segments, offspring_ids



print("=== Random Recombination Example ===")
grg_test = create_simple_grg() 

print(f"Loading: {GRG_FILE}")
print(f"Genome length: {grg_test.bp_range[1]}")
print(f"Initial samples: {grg_test.get_sample_nodes()}")
print()

display_grg(grg_test, "BEFORE random recombination")

genome = grg_test.bp_range
generations = 2

for gen in range(generations):
    print(f"\n=== Generation {gen+1} ===")

    parent_indices = np.arange(len(grg_test.get_sample_nodes()))
    np.random.shuffle(parent_indices)
    new_offspring = []

    samples = grg_test.get_sample_nodes()
    shuffled_samples = np.random.shuffle(samples)

    #offspring, segs = generate_offspring(grg_test, h1=0, num_offspring=2, h2=3)
    for i in range(0, len(samples), 2):

        p1 = samples[i]
        p2 = samples[i+1]

        # display_grg(grg_test, f"BEFORE random recombination {i+1}")

        print(f"Selected parents for offspring {i+1}: h1={p1}, h2={p2}")
        segments, offspring_ids = generate_offspring(grg_test, p1, p2, num_offspring=2, N=genome[1])
        new_offspring.extend(offspring_ids)

    new_offspring.sort()
    grg_test.set_samples(new_offspring)
    

# print()
# print(f"Generated offspring node: {offspring}")
# print(f"Samples (including offspring): {grg_test.get_sample_nodes()}")
# print(f"Segments inherited: {segs}")
# print()

# print("Segment breakdown:")
# start = 0
# for parent, end in segs:
#     print(f"  [{start}, {end}): from parent {parent}")
#     start = end

# Show after state
display_grg(grg_test, "AFTER random recombination")
